## Vocabulary overlap with RoBERTa's pretraining domain

### Key Imports

In [1]:
import sys

sys.path.insert(0, "..")
import pandas as pd
from huggingface_hub import hf_hub_download
from sklearn.feature_extraction.text import CountVectorizer

from data.apt_pools import build_pools
from data.loader_twd_filtered import fetch_filtered
from data.loader_wcb_unlabelled import fetch_sentences

### Vocabularies

In [2]:
# Gururangan et al. (2020) section 3.1. Their released code uses min_df=3 and the
# full vocabulary, which makes the Jaccard denominator depend on corpus length;
# a Wikipedia article carries far more terms than an FOMC sentence. The paper
# describes the top 10K unigrams instead, which fixes both vocabularies to the
# same size. We follow the paper, at 5,000 since our smallest corpus has 6,075.
N_DOCS = 50_000
TOP_K = 5_000


def vocab(texts):
    # stopwords excluded, as in their script
    vec = CountVectorizer(stop_words="english")
    # document-term matrix, so terms can be ranked by corpus frequency
    counts = vec.fit_transform(texts)
    # total occurrences of each term across the corpus
    freq = counts.sum(axis=0).A1
    # the TOP_K most frequent, so every vocabulary is the same size
    terms = vec.get_feature_names_out()
    return set(terms[freq.argsort()[::-1][:TOP_K]])


def overlap(a, b):
    # Jaccard: shared terms over all terms either vocabulary contains
    return 100 * len(a & b) / len(a | b)


### Corpora

In [3]:
# RoBERTa's own corpus is unreleased, so Gururangan sample sources similar to it.
# They use BookCorpus, Stories, Wikipedia and RealNews; we use Wikipedia alone.
wiki = pd.read_parquet(
    hf_hub_download(
        "wikimedia/wikipedia",
        "20231101.en/train-00000-of-00041.parquet",
        repo_type="dataset",
    )
)["text"].head(N_DOCS).tolist()

# the unfiltered FOMC scrape, the DAPT pool
fomc = build_pools(verbose=False)[0]["sentence"].head(N_DOCS).tolist()

# the keyword-filtered corpus, the task register
filtered = fetch_filtered()["sentence"].head(N_DOCS).tolist()

# 24 non-US central banks, the global pool's non-FOMC half
wcb = fetch_sentences()["sentence"].head(N_DOCS).tolist()

corpora = {"PT": wiki, "FOMC": fomc, "Filtered": filtered, "WCB": wcb}
for k, v in corpora.items():
    print(f"{k}: {len(v):,} documents")

downloading TDW repo tarball (~61MB)...


extracted -> C:\Users\vpati\AppData\Local\Temp\tmp8w2uvi7z


  meeting_minutes: 230 docs -> 47,340 sentences
  speech: 1026 docs -> 107,548 sentences
  press_conference: 63 docs -> 24,750 sentences


Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480


downloading TDW repo tarball (~61MB)...


...

Seed 5768 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480


downloading TDW repo tarball (~61MB)...


extracted -> C:\Users\vpati\AppData\Local\Temp\tmp262nz2z2
  meeting_minutes: 214 docs -> 20,618 sentences


  press_conference: 63 docs -> 5,086 sentences
  speech: 201 docs -> 12,465 sentences


PT: 50,000 documents
FOMC: 50,000 documents
Filtered: 35,257 documents
WCB: 50,000 documents


### Overlap

In [4]:
vocabs = {k: vocab(v) for k, v in corpora.items()}
for k, v in vocabs.items():
    print(f"{k}: {len(v):,} terms")

print()
rows = {a: {b: round(overlap(vocabs[a], vocabs[b]), 1) for b in vocabs} for a in vocabs}
print(pd.DataFrame(rows).to_string())

PT: 5,000 terms
FOMC: 5,000 terms
Filtered: 5,000 terms
WCB: 5,000 terms

             PT   FOMC  Filtered    WCB
PT        100.0   30.1      29.6   28.5
FOMC       30.1  100.0      70.6   53.1
Filtered   29.6   70.6     100.0   51.8
WCB        28.5   53.1      51.8  100.0
